# LeetCode 176 - Second Highest Salary (Practice Harness)

Run all cells after editing only `SOLUTION_SQL`.

Rules this harness enforces:
- Output must have exactly one column named `SecondHighestSalary`.
- Result must match expected output for every test case.
- `NULL` behavior is tested when a second distinct salary does not exist.


In [2]:
import duckdb

def run_case(solution_sql: str, rows: list[tuple[int, int | None]], case_name: str):
    con = duckdb.connect(':memory:')
    con.execute('CREATE TABLE Employee (id INT, salary INT);')

    if rows:
        con.executemany('INSERT INTO Employee (id, salary) VALUES (?, ?);', rows)

    actual = con.execute(solution_sql).fetchall()
    cols = [d[0] for d in con.description]

    if cols != ['SecondHighestSalary']:
        raise AssertionError(
            f"[{case_name}] Expected one column named 'SecondHighestSalary', got {cols}"
        )

    if len(actual) != 1:
        raise AssertionError(f"[{case_name}] Expected exactly 1 row, got {len(actual)}")

    return actual[0][0]


In [4]:
# Test fixtures: (case_name, employee_rows, expected_second_highest)
TEST_CASES = [
    (
        'basic_three_distinct',
        [(1, 100), (2, 200), (3, 300)],
        200,
    ),
    (
        'duplicates_at_top',
        [(1, 100), (2, 200), (3, 300), (4, 300)],
        200,
    ),
    (
        'all_same_salary',
        [(1, 100), (2, 100)],
        None,
    ),
    (
        'single_row_only',
        [(1, 999)],
        None,
    ),
    (
        'includes_negative',
        [(1, -5), (2, -1), (3, -10)],
        -5,
    ),
]


In [16]:
# ========================= EDIT ONLY THIS STRING =========================
SOLUTION_SQL = """
SELECT (SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1)  AS SecondHighestSalary
"""
# ======================================================================


In [17]:
results = []

for case_name, rows, expected in TEST_CASES:
    actual = run_case(SOLUTION_SQL, rows, case_name)
    passed = (actual == expected)
    results.append((case_name, expected, actual, passed))

print(f"{'case':30} {'expected':>10} {'actual':>10} {'pass':>6}")
print('-' * 64)
for case_name, expected, actual, passed in results:
    print(f"{case_name:30} {str(expected):>10} {str(actual):>10} {str(passed):>6}")

all_passed = all(r[3] for r in results)
print('\nOVERALL:', 'PASS' if all_passed else 'FAIL')

if not all_passed:
    failed = [r[0] for r in results if not r[3]]
    raise AssertionError(f'Failed cases: {failed}')


case                             expected     actual   pass
----------------------------------------------------------------
basic_three_distinct                  200        200   True
duplicates_at_top                     200        200   True
all_same_salary                      None       None   True
single_row_only                      None       None   True
includes_negative                      -5         -5   True

OVERALL: PASS


## EBNF of Bad SQL Query Shapes
```ebnf
bad_query         = n_plus_one | correlated_scan | unbounded_select | wildcard_select | cartesian_join ;
n_plus_one        = parent_query, repeated_child_query ;
parent_query      = "SELECT", columns, "FROM", table, [where_clause] ;
repeated_child_query = "FOR EACH", "parent_row", ":", "SELECT", columns, "FROM", table, "WHERE", fk, "=", parent_id ;
correlated_scan   = "SELECT", columns, "FROM", table, "WHERE", "EXISTS", "(", "SELECT", "1", "FROM", table, "WHERE", correlation_pred, ")" ;
unbounded_select  = "SELECT", columns, "FROM", table ;
wildcard_select   = "SELECT", "*", "FROM", table, [join_clause], [where_clause] ;
cartesian_join    = "SELECT", columns, "FROM", table, ",", table ;
columns           = "*" | column, {",", column} ;
where_clause      = "WHERE", predicate ;
join_clause       = "JOIN", table, "ON", predicate ;
predicate         = "..." ;
correlation_pred  = "child.col = parent.col" ;
table             = identifier ;
column            = identifier ;
fk                = identifier ;
parent_id         = identifier ;
identifier        = letter, { letter | digit | "_" } ;
```


## How to Fix Bad Query Shapes
- `N+1`: Replace per-row child lookups with one set-based query using `JOIN` + `GROUP BY`, or preload children with `WHERE fk IN (...)`.
- Correlated scan: Convert correlated subqueries to joins or CTEs so the child table is scanned once.
- Unbounded select: Add `WHERE`, `LIMIT`, and proper indexes for filter/sort columns.
- Wildcard select (`SELECT *`): Select only required columns to reduce I/O and payload size.
- Cartesian join: Always include explicit `JOIN ... ON ...` predicates.

### N+1 Example (Fix)
```sql
-- Bad: app runs one query for parents, then one query per parent for children (N+1)
-- Good: one query returns parent + aggregated child info
SELECT p.id, p.name, COUNT(c.id) AS child_count
FROM parent p
LEFT JOIN child c ON c.parent_id = p.id
GROUP BY p.id, p.name;
```


## Usage
1. Change only `SOLUTION_SQL`.
2. Run all cells.
3. Fix query until `OVERALL: PASS`.
